# Failure Attribution
# 0. 介绍

**研究背景**：Agent 的一次运行不是只有“输入问题、输出答案”两步，而是模型连续读取上下文、选择工具、改变环境并结束任务的完整过程。最终结果由模型、工具、上下文、执行环境、生命周期、观测、评估器和权限共同决定；Verification（V）层的职责，是把这次运行变成可复核的证据，并回答“任务是否完成”和“失败应修哪里”。

**现存问题**：生产中常见的错误基线只保存最终的成功/失败，或者看到环境没有改变就直接写成“模型失败”。例如工具超时、权限拒绝、沙箱依赖不一致、上下文丢失、生命周期提前停止或评分器本身不稳定，都可能让一个正确的模型决定得到失败结果；如果没有工具回执、状态变化、停止原因和完整 trace，这些故障会被混在一起，团队只能盲目修改 Prompt，既修不好真正的问题，也无法比较修复是否有效。单次成功率还会掩盖执行路径的浪费、违规和随机波动。

**解决方案**：本 Notebook 实现`trace-native evaluation + 多层判定 + 分层失败归因`：先固定任务、环境、工具、权限和成功标准，再做执行前 readiness 检查并捕获完整轨迹；随后分别检查`结果层（Outcome）`、`轨迹层（Trajectory）`和`评估器层（Evaluator）`，最后依据证据把失败定位到 E/T/C/L/O/V/G 或模型，并生成可执行的修复建议。这个方向与 SWE-bench、Terminal-Bench、OSWorld 的状态/测试验证、HAL 与 Repo2Run 的评估基础设施、Anthropic Evals、LangChain Deep-Agent Evaluation、promptfoo 和 DeepEval 的持续评估实践一致。改进版不更换模型，而是保留同一 rollout 的工具回执、权限决策、状态 diff、stop reason、Token、成本、延迟和 grader 版本，并用确定性检查器、校准后的 LLM judge 及高风险人工复核组成证据链，让一次“失败”变成可以复现、归因和回归验证的工程诊断。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义错误基线组件 *
5. 展示基线故障 *
6. 定义三层判定与失败归因 *
7. 展示修复结果 *
8. 汇总消融对照


# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定任务与成功标准
失败归因必须先明确任务到底要求什么，否则连“是否失败”都无法判断。下面固定一个生产中常见的服务回滚任务：只有订单服务最终变成目标版本，任务才算成功；模型口头声称完成不算。

In [2]:
# target_version 是两条实验路径共用的唯一正确终态
# request 只描述任务，不提前告诉模型失败会发生在哪里
task = {
    "service": "orders",
    "target_version": "v1.8.2",
    "request": "请把订单服务回滚到 v1.8.2。",
}

print("任务：", task["request"])
print("成功标准：服务版本变为", task["target_version"])

任务： 请把订单服务回滚到 v1.8.2。
成功标准：服务版本变为 v1.8.2


输出给出了唯一的任务和成功标准。后续基线与改进版本都会使用这两项固定内容，不会因为归因方法不同而更换题目；下一步准备任务开始前的真实状态。

## 2.2 固定初始执行状态
同一次失败可能来自模型，也可能来自模型外部。下面把服务初始版本和回滚审批状态写成明确数据：版本尚未回滚，而且生产审批没有通过。这是生产自动化中常见的失败条件，后续工具会依据它返回事实。

In [3]:
# version 表示工具执行前可以直接观察的环境状态
# rollback_approved=False 表示生产回滚尚未获得审批
initial_state = {
    "version": "v1.9.0",
    "rollback_approved": False,
}

print("初始版本：", initial_state["version"])
print("回滚审批：", initial_state["rollback_approved"])

初始版本： v1.9.0
回滚审批： False


输出显示服务仍是 `v1.9.0`，并且回滚审批为 `False`。这些是后续判断责任来源的环境事实，但此时工具尚未执行；下一步把回滚能力说明给真实模型。

## 2.3 说明工具调用格式
大模型不能直接操作部署系统，只能提交结构化工具请求。下面用 JSON Schema 告诉模型工具名称，以及必须填写的服务名和目标版本；工具真正执行后的回执会在后文保留为归因证据。

In [4]:
# service 指定要操作的服务，version 指定要回滚到的版本
# required 让模型必须同时提交这两个核心参数
tools = [{
    "type": "function",
    "function": {
        "name": "rollback_service",
        "description": "把指定服务回滚到目标版本",
        "parameters": {
            "type": "object",
            "properties": {
                "service": {"type": "string"},
                "version": {"type": "string"},
            },
            "required": ["service", "version"],
        },
    },
}]

print("工具名称：", tools[0]["function"]["name"])
print("必填参数：", tools[0]["function"]["parameters"]["required"])

工具名称： rollback_service
必填参数： ['service', 'version']


输出说明模型只能请求 `rollback_service`，并且必须给出 `service` 与 `version`。工具格式已经固定，下一步组装本次真实 API 请求要读取的消息。

## 2.4 组装模型输入
为了判断模型本身是否理解任务，系统消息只要求它使用回滚工具，不向它透露审批状态。这样，模型负责选择动作，工具负责返回外部事实，两者的责任边界清楚。

In [5]:
# system 消息规定模型必须通过工具提出回滚动作
# user 消息直接复用刚才固定的任务，避免两条路径输入不同
messages = [
    {
        "role": "system",
        "content": "你是部署助手。必须调用 rollback_service 完成回滚，不要猜测执行结果。",
    },
    {"role": "user", "content": task["request"]},
]

print("系统要求：", messages[0]["content"])
print("用户请求：", messages[1]["content"])

系统要求： 你是部署助手。必须调用 rollback_service 完成回滚，不要猜测执行结果。
用户请求： 请把订单服务回滚到 v1.8.2。


输出展示了真实模型即将看到的完整输入。至此，任务、成功标准、初始状态、工具和消息都已固定；下一章将发送真实 API 请求，并读取模型实际生成的工具调用。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
任务和工具已经准备完成。下面把它们发送给 `.env` 指定的真实模型，并要求模型必须选择工具；同时记录本次请求的实际等待时间。

In [6]:
from time import perf_counter

# 计时范围只覆盖本次真实 API 请求
# tool_choice="required" 要求模型返回结构化工具调用
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)

print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实 API 已经返回，完整响应保存在 `response` 中。此时模型只提出了动作，回滚工具还没有执行；下一步读取工具名称和具体参数。

## 3.2 读取模型动作
工具调用把模型决定变成了可直接查看的字段。下面取出工具名称，并把 JSON 参数还原为 Python 字典；后续两条实验路径会共享这份真实模型动作。

In [7]:
import json

# 第一条 choice 保存本次真实请求返回的模型决定
# arguments 是 JSON 字符串，需要还原成可读取的字典
choice = response.choices[0]
tool_call = choice.message.tool_calls[0]
model_action = {
    "tool": tool_call.function.name,
    "arguments": json.loads(tool_call.function.arguments),
}

print("工具：", model_action["tool"])
print("参数：", model_action["arguments"])

工具： rollback_service
参数： {'service': '订单服务', 'version': 'v1.8.2'}


输出展示了真实模型选择的工具与参数。这些字段只说明模型想做什么，不能说明外部系统是否允许或完成了操作；下一步保存本次请求的运行信息。

## 3.3 保存请求信息
失败归因还需要知道这次动作来自哪个模型、消耗多少 Token、等待多久，以及模型为何停止。下面把真实响应中的这些字段放进同一个字典，供后续报告直接使用。

In [8]:
# Token 和停止原因直接读取真实 API 响应
# provider 没有返回计费金额，因此成本明确保存为 None
usage = response.usage
api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": api_latency_ms,
    "stop_reason": choice.finish_reason,
}

print(api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 184, 'output_tokens': 111, 'total_tokens': 295, 'cost_usd': None, 'latency_ms': 2547, 'stop_reason': 'tool_calls'}


输出记录了本次真实调用的 provider、model、Token、成本、延迟和停止原因。`tool_calls` 只表示模型已经提交工具请求，不表示任务成功；下一章将定义只看最终环境状态的错误基线组件。

# 4. 定义错误基线组件
## 4.1 定义回滚工具
要判断失败来自哪里，必须先让模型动作进入外部系统。下面定义一个最小回滚工具：审批通过时更新版本，审批未通过时保持原状态并返回 `permission_denied`。这份工具回执是真正的执行事实。

In [9]:
# 复制初始状态，避免工具直接覆盖第 2 章的共同起点
# 工具根据审批状态决定是否更新目标版本
def rollback_service(arguments, state):
    final_state = state.copy()

    if state["rollback_approved"]:
        final_state["version"] = arguments["version"]
        tool_result = {"ok": True, "status": "completed"}
    else:
        tool_result = {
            "ok": False,
            "status": "permission_denied",
            "message": "生产回滚需要审批",
        }

    return final_state, tool_result

print("回滚工具已定义")

回滚工具已定义


输出说明回滚工具已经定义，但还没有执行。它会同时返回最终环境状态和工具回执；下一步定义生产中常见的错误基线，让它故意丢弃工具回执。

## 4.2 定义只看最终结果的报告器
错误基线只比较最终版本与目标版本：版本不一致，就写成“模型失败”。它不接收模型动作、工具回执或审批状态，因此任何外部故障都会被压缩成同一个错误标签。

In [10]:
# task_success 只由最终版本是否达到目标决定
# 失败时直接归因给 model，故意复现错误生产基线
def create_baseline_report(final_state, target_version):
    task_success = final_state["version"] == target_version

    if task_success:
        failure_source = None
        reason = "目标版本已经生效"
    else:
        failure_source = "model"
        reason = "目标版本没有生效"

    return {
        "task_success": task_success,
        "failure_source": failure_source,
        "reason": reason,
    }

print("错误基线报告器已定义")

错误基线报告器已定义


输出说明错误基线已经准备好，但尚未生成报告。它的输入合同已经暴露了根本缺陷：只接收最终状态，不接收执行过程；下一章将运行工具，并观察它如何把明确的权限拒绝误报为模型失败。

# 5. 展示基线故障
## 5.1 执行真实模型动作
下面把第 3 章真实模型生成的动作交给回滚工具。模型动作、工具回执和版本变化会同时打印出来，使执行失败发生在哪一步一目了然。

In [11]:
# arguments 来自本次真实 API 返回的工具参数
# initial_state 是两条实验路径共用的同一初始状态
baseline_final_state, tool_result = rollback_service(
    model_action["arguments"],
    initial_state,
)

print("模型动作：", model_action)
print("工具回执：", tool_result)
print("版本变化：", initial_state["version"], "->", baseline_final_state["version"])

模型动作： {'tool': 'rollback_service', 'arguments': {'service': '订单服务', 'version': 'v1.8.2'}}
工具回执： {'ok': False, 'status': 'permission_denied', 'message': '生产回滚需要审批'}
版本变化： v1.9.0 -> v1.9.0


输出显示真实模型选择了回滚工具和目标版本 `v1.8.2`，但工具因缺少审批返回 `permission_denied`，版本仍是 `v1.9.0`。失败原因已经出现在工具回执中；下一步观察错误基线如何报告同一次运行。

## 5.2 生成错误基线报告
现在只把最终状态交给基线报告器，不传入刚才的模型动作和工具回执。这正是生产中“只看结果”的错误做法：过程证据在进入报告前就被丢掉了。

In [12]:
# 报告器只能看到未达到目标的最终版本
# tool_result 没有传入，因此权限拒绝不会出现在报告中
baseline_report = create_baseline_report(
    baseline_final_state,
    task["target_version"],
)

print("基线报告：", baseline_report)

基线报告： {'task_success': False, 'failure_source': 'model', 'reason': '目标版本没有生效'}


输出把失败来源写成了 `model`，但上一格的直接证据是 `permission_denied`。同一次运行出现“工具明确拒绝、报告却归咎模型”的矛盾，证明只看最终结果无法完成可靠归因；下一章将定义结果层、轨迹层和评估器层三层判定，并让工具回执进入诊断。

# 6. 定义三层判定与失败归因
## 6.1 定义完整运行轨迹
改进版本首先停止丢弃过程证据。下面把输入任务、真实模型动作、工具回执、状态变化和 API 指标放进同一条 trace；后续所有判断都读取这份事实记录，而不是只看最后一个版本号。

In [13]:
# trace 按执行顺序保存输入、动作、工具结果和环境变化
# runtime 保留真实请求的模型、Token、延迟与停止原因
def build_trace(task, action, result, before, after, metrics):
    return {
        "input": {
            "request": task["request"],
            "target_version": task["target_version"],
        },
        "model_action": action,
        "tool_result": result,
        "state_diff": {
            "version": {
                "before": before["version"],
                "after": after["version"],
            },
        },
        "runtime": metrics.copy(),
    }

print("运行轨迹组件已定义")

运行轨迹组件已定义


输出说明 trace 组件已经定义，但尚未组装数据。它会保留从输入到最终状态的完整因果链；下一步定义只回答“目标是否完成”的 Outcome 判定。

## 6.2 定义结果层判定
Outcome 只判断最终状态是否达到任务目标，不负责解释原因。这个边界很重要：结果失败是事实，但仅凭这个事实不能推出模型失败。

In [14]:
# after 是工具执行后的实际版本，target_version 是预期版本
# passed 只表达任务结果，不携带任何责任归因
def grade_outcome(trace):
    observed = trace["state_diff"]["version"]["after"]
    expected = trace["input"]["target_version"]
    passed = observed == expected

    return {
        "level": "outcome",
        "passed": passed,
        "evidence": {"observed": observed, "expected": expected},
    }

print("Outcome 判定器已定义")

Outcome 判定器已定义


输出说明 Outcome 判定器已经定义。它只会确认版本是否改变，不会像错误基线那样直接指定责任；下一步检查模型选择的执行路径是否正确。

## 6.3 定义轨迹层判定
Trajectory 检查模型是否选择了正确工具和目标版本，并保留工具最终返回的状态。这样可以区分“模型动作错误”和“模型动作正确但外部系统拒绝”。

In [15]:
# action_correct 同时检查工具名称和任务要求的目标版本
# tool_status 作为独立证据保留，不混进模型动作评分
def grade_trajectory(trace):
    action = trace["model_action"]
    expected = trace["input"]["target_version"]
    action_correct = (
        action["tool"] == "rollback_service"
        and action["arguments"]["version"] == expected
    )

    return {
        "level": "trajectory",
        "passed": action_correct,
        "evidence": {
            "tool": action["tool"],
            "version": action["arguments"]["version"],
            "tool_status": trace["tool_result"]["status"],
        },
    }

print("Trajectory 判定器已定义")

Trajectory 判定器已定义


输出说明 Trajectory 判定器已经定义。模型动作和工具回执现在是两个独立信号：前者衡量路径，后者说明外部执行发生了什么；下一步检查判定器本身是否稳定。

## 6.4 定义评估器层判定
Evaluator 不能被当作永远正确的裁判。这个任务的最终状态可以精确比较，因此下面用同一个确定性 Outcome 判定结果重复两次；两次结论一致，才说明评分信号稳定。

In [16]:
# first 与 second 来自同一 trace 的两次独立 Outcome 判定
# stable 表示同一输入是否得到一致的任务结论
def grade_evaluator(first, second):
    scores = [first["passed"], second["passed"]]
    stable = scores[0] == scores[1]

    return {
        "level": "evaluator",
        "passed": stable,
        "evidence": {"repeated_scores": scores},
    }

print("Evaluator 判定器已定义")

Evaluator 判定器已定义


输出说明 Evaluator 判定器已经定义。对于可直接比较的环境状态，确定性检查比额外语义评审更简单、更稳定；下一步把三层结果与工具证据合成可行动的归因。

## 6.5 定义失败归因器
归因器先确认任务确实失败，再按直接证据定位责任：评分不稳定指向 V 层，权限拒绝指向 G 层，模型动作错误指向模型。没有充分证据时保留 `unknown`，避免制造新的错误结论。

In [17]:
# 规则按结果、评估器、工具回执和模型路径依次读取证据
# 每个归因同时返回原因和下一步动作，便于直接修复
def attribute_failure(trace, outcome, trajectory, evaluator):
    if outcome["passed"]:
        source = None
        reason = "任务已经完成"
        next_action = "无需修复"
    elif not evaluator["passed"]:
        source = "V"
        reason = "评分结果不稳定"
        next_action = "先修复评分器"
    elif trace["tool_result"]["status"] == "permission_denied":
        source = "G"
        reason = trace["tool_result"]["message"]
        next_action = "完成生产回滚审批后重试"
    elif not trajectory["passed"]:
        source = "model"
        reason = "模型选择了错误动作"
        next_action = "检查任务理解与工具选择"
    else:
        source = "unknown"
        reason = "当前证据不足"
        next_action = "补充缺失的运行证据"

    return {
        "failure_source": source,
        "reason": reason,
        "next_action": next_action,
    }

print("失败归因器已定义")

失败归因器已定义


输出说明完整改进链已经定义，但本章尚未运行归因。它不会改变模型动作或绕过审批，只会保留证据并修正责任判断；下一章将用第 5 章的同一次失败运行这条链路。

# 7. 展示修复结果
## 7.1 组装完整运行轨迹
先把第 5 章同一次失败的全部事实放进 trace。这里不重新调用模型或工具，只整理已经发生的输入、动作、工具回执、状态变化和运行指标。

In [18]:
# 所有参数都来自本次从头执行已经产生的真实数据
# trace 只整理证据，不改变模型动作或环境状态
failure_trace = build_trace(
    task,
    model_action,
    tool_result,
    initial_state,
    baseline_final_state,
    api_metrics,
)

print("输入上下文：", failure_trace["input"])
print("模型动作：", failure_trace["model_action"])
print("工具回执：", failure_trace["tool_result"])
print("状态变化：", failure_trace["state_diff"])
print("运行信息：", failure_trace["runtime"])

输入上下文： {'request': '请把订单服务回滚到 v1.8.2。', 'target_version': 'v1.8.2'}
模型动作： {'tool': 'rollback_service', 'arguments': {'service': '订单服务', 'version': 'v1.8.2'}}
工具回执： {'ok': False, 'status': 'permission_denied', 'message': '生产回滚需要审批'}
状态变化： {'version': {'before': 'v1.9.0', 'after': 'v1.9.0'}}
运行信息： {'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 184, 'output_tokens': 111, 'total_tokens': 295, 'cost_usd': None, 'latency_ms': 2547, 'stop_reason': 'tool_calls'}


输出把输入上下文、模型决定、权限回执、状态 diff 和真实 API 指标连接成一条证据链。与基线相比，失败发生前后的事实不再丢失；下一步只判断最终任务是否完成。

## 7.2 运行结果层判定
Outcome 只比较实际版本和目标版本。它应当诚实地给出失败，但不会把这个结果直接解释成模型错误。

In [19]:
# Outcome 读取 trace 中的最终版本和目标版本
# 本格只生成任务结果，不生成责任归因
outcome_result = grade_outcome(failure_trace)

print("Outcome：", outcome_result)

Outcome： {'level': 'outcome', 'passed': False, 'evidence': {'observed': 'v1.9.0', 'expected': 'v1.8.2'}}


输出中的 `passed=False` 说明回滚任务确实没有完成，实际版本仍是 `v1.9.0`。这只回答“结果失败”，没有回答“谁导致失败”；下一步检查模型选择的路径。

## 7.3 运行轨迹层判定
Trajectory 会检查模型是否选择了回滚工具和正确目标版本，同时展示工具返回的状态。如果模型动作正确而工具拒绝，两个信号必须分开呈现。

In [20]:
# Trajectory 评分只判断模型选择的工具与目标版本
# permission_denied 作为独立工具证据保留在结果中
trajectory_result = grade_trajectory(failure_trace)

print("Trajectory：", trajectory_result)

Trajectory： {'level': 'trajectory', 'passed': True, 'evidence': {'tool': 'rollback_service', 'version': 'v1.8.2', 'tool_status': 'permission_denied'}}


输出中的 `passed=True` 说明模型选择了正确工具和 `v1.8.2`，而 `tool_status=permission_denied` 说明阻断发生在模型动作之后。下一步检查 Outcome 判定器能否稳定重复同一结论。

## 7.4 运行评估器层判定
### 7.4.1 重复结果层判定
先对完全相同的 trace 再运行一次 Outcome 判定。这个单元只生成第二份结果，下一格再比较两次评分。

In [21]:
# 输入仍是同一条 failure_trace，没有改变任何任务事实
# repeated_outcome 单独保存第二次确定性判定结果
repeated_outcome = grade_outcome(failure_trace)

print("重复 Outcome：", repeated_outcome)

重复 Outcome： {'level': 'outcome', 'passed': False, 'evidence': {'observed': 'v1.9.0', 'expected': 'v1.8.2'}}


输出再次得到 `passed=False`，与第一次 Outcome 相同。两次原始评分已经准备好；下一步只比较它们是否一致。

### 7.4.2 比较两次评分
现在把两次 Outcome 结果交给 Evaluator 判定器。它不重新判断任务，只检查同一输入是否产生一致结论。

In [22]:
# first 和 second 分别是刚才两次独立的 Outcome 结果
# Evaluator 只比较评分稳定性，不参与失败责任判断
evaluator_result = grade_evaluator(outcome_result, repeated_outcome)

print("Evaluator：", evaluator_result)

Evaluator： {'level': 'evaluator', 'passed': True, 'evidence': {'repeated_scores': [False, False]}}


输出中的 `passed=True` 和 `[False, False]` 表示评分器对同一状态给出稳定结论，因此本次失败不是评分波动造成的。三层结果都已得到；下一步生成最终诊断。

## 7.5 生成正确归因
最后把完整 trace 与三层结果交给归因器。它应当保留“任务失败、模型路径正确、评分稳定、工具因审批拒绝”这条因果链，并输出真正需要修复的位置。

In [23]:
# 归因器同时读取结果、路径、评分稳定性和工具直接证据
# attribution 保存责任层、原因和可以立即执行的下一步
attribution = attribute_failure(
    failure_trace,
    outcome_result,
    trajectory_result,
    evaluator_result,
)

print("正确归因：", attribution)

正确归因： {'failure_source': 'G', 'reason': '生产回滚需要审批', 'next_action': '完成生产回滚审批后重试'}


输出把失败来源从基线的 `model` 修正为 Governance（`G`）层，并给出“完成生产回滚审批后重试”的行动建议。回滚仍然失败是正确行为，因为审批尚未补齐；本章修复的是诊断准确性，而不是绕过外部约束。下一章将汇总前后消融对照。

# 8. 汇总消融对照
## 8.1 准备同源对照数据
最后比较无、有三层失败归因的两条报告路径。两者共享同一次真实 API rollout、同一个模型动作、同一份工具回执和同一个失败结果；唯一变化是 Harness 是否保留完整 trace 并分别判断 Outcome、Trajectory 与 Evaluator。

In [24]:
# G 是 permission_denied 直接证据对应的正确责任层
# 两行复用同一次 API 指标，确保只消融失败归因机制
expected_failure_source = "G"
comparison_rows = [
    {
        "方案": "无三层归因",
        "任务成功": baseline_report["task_success"],
        "判定层级": "Outcome",
        "报告责任": baseline_report["failure_source"],
        "归因正确": baseline_report["failure_source"] == expected_failure_source,
        "可行动建议": False,
        "API 调用": 1,
        "Token": api_metrics["total_tokens"],
        "成本 USD": api_metrics["cost_usd"],
        "API 延迟 ms": api_metrics["latency_ms"],
    },
    {
        "方案": "有三层归因",
        "任务成功": outcome_result["passed"],
        "判定层级": "Outcome + Trajectory + Evaluator",
        "报告责任": attribution["failure_source"],
        "归因正确": attribution["failure_source"] == expected_failure_source,
        "可行动建议": bool(attribution["next_action"]),
        "API 调用": 1,
        "Token": api_metrics["total_tokens"],
        "成本 USD": api_metrics["cost_usd"],
        "API 延迟 ms": api_metrics["latency_ms"],
    },
]

print("对照数据已准备：", len(comparison_rows), "种方案")

对照数据已准备： 2 种方案


输出说明两种方案的对照数据已经准备好。每行都引用前面真实运行保存的结果，没有发起第二次模型请求；下一步把关键指标放进同一张表。

## 8.2 展示消融结果
下面直接展示任务结果、判定层级、责任归因、行动建议和 API 开销。任务成功不应因诊断器而改变；真正需要改善的是归因是否正确、报告是否能指导修复。

In [25]:
import pandas as pd

# DataFrame 只负责把两行同源结果排成可比较的表格
# 所有数值都来自本次从头执行已经保存的变量
comparison_table = pd.DataFrame(comparison_rows)

display(comparison_table)

,方案,任务成功,判定层级,报告责任,归因正确,可行动建议,API 调用,Token,成本 USD,API 延迟 ms
0,无三层归因,False,Outcome,model,False,False,1,295,None,2547
1,有三层归因,False,Outcome + Trajectory + Evaluator,G,True,True,1,295,None,2547


两行的任务结果都是失败，API 调用、Token、成本和延迟也完全相同，因为它们共享同一次真实 rollout。无三层归因时，报告把权限拒绝错写成 `model`；加入 trace 与三层判定后，归因变为正确的 `G`，并给出可执行建议。单样本归因准确率因此从 `0%` 变为 `100%`，且没有增加模型调用。金额仍为 `None`，因为 provider 没有返回计费金额；本表没有用猜测价格替代真实数据。

## 8.6 拓展

### nano 版省略了什么

本 Notebook 只复现单次、单工具、单一直接原因的主线。生产版本还需要处理多原因归因、长轨迹 span 树、跨服务事件关联、评分器版本管理、重复 rollout 方差、人工复核、归因置信度，以及把失败 trace 自动转成回归用例。这些能力不会改变本例的核心原则：先保留事实，再分层判断，最后只做有证据支持的归因。

### 延伸阅读

1. 2026, [AgentFixer](https://arxiv.org/abs/2603.29848)：从失败检测、证据定位到可执行修复建议。
2. 2026, [AgentDoG](https://arxiv.org/abs/2601.18491)：面向 Agent 安全与可靠性的诊断护栏。
3. 2026, [Anthropic, Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)：用 transcript、grader 和失败类别连接现象与责任层。